# Disease Incidence Trend Analysis

## Project Overview
Analyze public health incidence data over time and geography to identify trends, seasonal patterns, and outlier regions.

## Learning Objectives
- Calculate incidence rates per 100,000 population across regions and time
- Identify seasonal patterns in disease incidence
- Detect outlier regions with unusually high or low rates
- Analyze multi-year trends for epidemiological insight

## Problem Statement
Public health officials need to monitor disease trends to allocate resources, detect outbreaks early, and evaluate intervention effectiveness across regions.

## Why This Project Matters
Disease surveillance analytics enable early outbreak detection, resource allocation, and evidence-based public health policy. Timely trend analysis saves lives.

## Dataset Overview
Synthetic disease incidence dataset: monthly case counts for 5 diseases across 10 regions over 5 years (2019-2023).

## Dataset Source and License Notes
- Synthetic data inspired by CDC WONDER-style reporting
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
regions = ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West Coast',
           'Mountain', 'Great Plains', 'Mid-Atlantic', 'Gulf Coast', 'Pacific Northwest']
diseases = ['Influenza', 'COVID-19', 'Salmonella', 'Lyme Disease', 'Hepatitis C']
months = pd.date_range('2019-01-01', '2023-12-01', freq='MS')
pop = {r: np.random.randint(2_000_000, 15_000_000) for r in regions}

rows = []
for disease in diseases:
    for region in regions:
        base = {'Influenza': 120, 'COVID-19': 80, 'Salmonella': 30, 'Lyme Disease': 15, 'Hepatitis C': 25}[disease]
        region_mult = np.random.uniform(0.6, 1.5)
        for m in months:
            seasonal = 1.0
            if disease == 'Influenza':
                seasonal = 1 + 0.8 * np.cos(2 * np.pi * (m.month - 1) / 12)
            elif disease == 'COVID-19':
                if m.year < 2020:
                    seasonal = 0.01
                elif m.year == 2020:
                    seasonal = 0.5 + 2.0 * (m.month >= 3)
                else:
                    seasonal = 1.5 + 0.5 * np.sin(2 * np.pi * m.month / 12)
            elif disease == 'Salmonella':
                seasonal = 1 + 0.5 * np.sin(2 * np.pi * (m.month - 4) / 12)
            elif disease == 'Lyme Disease':
                seasonal = 1 + 1.2 * np.exp(-0.5 * ((m.month - 7) / 2) ** 2)

            cases = max(0, int(base * region_mult * seasonal * pop[region] / 5_000_000 *
                               np.random.lognormal(0, 0.25)))
            rows.append({
                'Disease': disease, 'Region': region, 'Month': m,
                'Cases': cases, 'Population': pop[region],
            })

df = pd.DataFrame(rows)
df['IncidenceRate'] = (df['Cases'] / df['Population'] * 100_000).round(2)
df['Year'] = df['Month'].dt.year
df['MonthNum'] = df['Month'].dt.month

print(f'Dataset: {df.shape}')
print(f'Diseases: {df["Disease"].nunique()}, Regions: {df["Region"].nunique()}')
print(f'Date range: {df["Month"].min()} to {df["Month"].max()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nCases stats by disease:')
print(df.groupby('Disease')['Cases'].agg(['sum','mean','max']).round(0))

## Overall Incidence Trends

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for disease in diseases:
    sub = df[df['Disease'] == disease].groupby('Month')['IncidenceRate'].mean()
    axes[0].plot(sub.index, sub.values, label=disease, linewidth=1.5)
axes[0].set_title('Average Incidence Rate Over Time (per 100K)')
axes[0].legend()

annual = df.groupby(['Year', 'Disease'])['Cases'].sum().unstack()
annual.plot.bar(ax=axes[1], edgecolor='black')
axes[1].set_title('Annual Total Cases by Disease')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'overall_trends.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonal Patterns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, disease in enumerate(diseases):
    sub = df[df['Disease'] == disease]
    seasonal = sub.groupby('MonthNum')['IncidenceRate'].mean()
    axes[i].bar(seasonal.index, seasonal.values, edgecolor='black', color='steelblue')
    axes[i].set_title(f'{disease} — Monthly Pattern')
    axes[i].set_xlabel('Month')
    axes[i].set_xticks(range(1, 13))
axes[5].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'seasonal_patterns.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Comparison

In [ ]:
region_rates = df.groupby(['Region', 'Disease'])['IncidenceRate'].mean().unstack().round(2)
print('Avg Incidence Rate by Region × Disease:')
print(region_rates)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(region_rates, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('Avg Incidence Rate per 100K: Region × Disease')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Outlier Region Detection

In [ ]:
overall_mean = df.groupby('Disease')['IncidenceRate'].mean()
overall_std = df.groupby('Disease')['IncidenceRate'].std()
region_mean = df.groupby(['Region', 'Disease'])['IncidenceRate'].mean().unstack()

z_scores = (region_mean - overall_mean) / overall_std
print('Z-Scores by Region (positive = above average):')
print(z_scores.round(2))

outliers = z_scores.abs() > 1.5
print(f'\nOutlier region-disease combinations (|z| > 1.5):')
for r in outliers.index:
    for d in outliers.columns:
        if outliers.loc[r, d]:
            print(f'  {r} × {d}: z={z_scores.loc[r, d]:.2f}')

## COVID-19 Deep Dive

In [ ]:
covid = df[df['Disease'] == 'COVID-19']
fig, ax = plt.subplots(figsize=(14, 5))
for region in ['Northeast', 'Southeast', 'West Coast', 'Midwest']:
    sub = covid[covid['Region'] == region]
    ax.plot(sub['Month'], sub['IncidenceRate'], label=region, alpha=0.8)
ax.set_title('COVID-19 Incidence Rate by Region')
ax.set_ylabel('Per 100K')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'covid_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Influenza** shows classic winter seasonality — peak Dec-Feb, trough summer
- **COVID-19** shows the expected emergence pattern (near-zero pre-2020, surge in 2020, then wave pattern)
- **Salmonella** peaks in summer months — consistent with foodborne transmission
- **Lyme Disease** peaks in summer (June-August) — consistent with tick season
- **Regional variation** is significant — geography, climate, and population density drive differences
- **Outlier detection** identifies regions needing investigation or targeted public health intervention

## Limitations
- Synthetic data with programmed seasonal patterns
- No demographic stratification (age, race, socioeconomic)
- No intervention or vaccination data to explain trend changes
- Population denominators are static (no year-over-year growth)
- No reporting delay or undercount modeling

## How to Improve This Project
- Add demographic stratification for health equity analysis
- Include vaccination coverage as an explanatory variable
- Add outbreak detection algorithms (CUSUM, Farrington)
- Include reporting confidence intervals
- Link to intervention timelines for causal analysis

## Production Considerations
- Weekly automated surveillance reports for public health agencies
- Real-time outbreak detection alerting
- Interactive geographic dashboards (choropleth maps)
- Integration with lab reporting systems for complete case capture

## Common Mistakes
- Comparing raw case counts without normalizing by population
- Ignoring seasonal baselines when declaring outbreaks
- Not accounting for reporting delays and testing capacity changes
- Treating all regions as comparable without considering demographic differences

## Mini Challenge / Exercises
1. Calculate the year-over-year % change in Influenza incidence for each region.
2. Which region experienced the largest COVID-19 surge in 2020?
3. Build a simple seasonal index for Salmonella (monthly avg / annual avg).
4. Identify the month with the highest aggregate disease burden across all diseases.

## Final Summary / Key Takeaways
- Disease incidence analysis reveals temporal and geographic patterns critical for public health
- Seasonal patterns vary by disease and must inform resource planning
- Regional outliers merit investigation — they may indicate outbreaks, reporting issues, or genuine differences
- Population-normalized rates (per 100K) are essential for fair comparison
- Surveillance analytics are the foundation of evidence-based public health response